# bc-pong — Behavioral Cloning Training
This notebook walks through loading gameplay data, inspecting it, and training a mimicker MLP.

## 1. Load all session CSVs

In [36]:
import pandas as pd
import numpy as np
import glob
import os

DATA_DIR = os.path.join("..", "bc-pong/data")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "session_*.csv")))
print(f"Found {len(csv_files)} session files:")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

Found 10 session files:
  session_20260419_111656.csv
  session_20260419_112537.csv
  session_20260419_184451.csv
  session_20260419_184638.csv
  session_20260419_185017.csv
  session_20260420_180102.csv
  session_20260420_180545.csv
  session_20260420_184858.csv
  session_20260420_185314.csv
  session_20260420_185514.csv


In [37]:
# load and tag with session
frames = []
for i, path in enumerate(csv_files):
    df = pd.read_csv(path)
    df["session"] = i
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

print(f"\nTotal rows : {len(data):,}")
print(f"Sessions   : {data['session'].nunique()}")
print(f"Columns    : {list(data.columns)}")
data.head(10)


Total rows : 100,498
Sessions   : 10
Columns    : ['ball_x', 'ball_y', 'ball_vx', 'ball_vy', 'left_y', 'right_y', 'left_action', 'right_action', 'left_scored', 'right_scored', 'session']


,ball_x,ball_y,ball_vx,ball_vy,left_y,right_y,left_action,right_action,left_scored,right_scored,session
0,404.624,301.901,4.624,1.901,260,260,0,0,0,0,0
1,409.249,303.802,4.624,1.901,260,260,0,0,0,0,0
2,413.873,305.703,4.624,1.901,260,260,0,0,0,0,0
3,418.498,307.605,4.624,1.901,260,260,0,0,0,0,0
4,423.122,309.506,4.624,1.901,260,260,0,0,0,0,0
5,427.747,311.407,4.624,1.901,260,260,0,0,0,0,0
6,432.371,313.308,4.624,1.901,260,260,0,0,0,0,0
7,436.996,315.209,4.624,1.901,260,260,0,0,0,0,0
8,441.620,317.110,4.624,1.901,260,260,0,0,0,0,0
9,446.245,319.012,4.624,1.901,260,260,0,0,0,0,0


In [38]:
# sanity check
print("Null counts:")
print(data.isnull().sum())
print()
print("Value ranges:")
print(data.describe().round(2))

Null counts:
ball_x          0
ball_y          0
ball_vx         0
ball_vy         0
left_y          0
right_y         0
left_action     0
right_action    0
left_scored     0
right_scored    0
session         0
dtype: int64

Value ranges:
          ball_x     ball_y    ball_vx    ball_vy     left_y    right_y  \
count  100498.00  100498.00  100498.00  100498.00  100498.00  100498.00   
mean      402.64     288.67       0.05       0.03     259.04     263.55   
std       206.75     161.57       5.66       3.62     132.41     146.78   
min        -5.96       6.00     -10.96      -9.79       0.00       0.00   
25%       224.08     152.75      -5.34      -2.94     165.00     150.00   
50%       404.25     291.97       2.65       0.08     260.00     265.00   
75%       581.18     415.26       5.29       2.90     360.00     380.00   
max       805.96     594.00      10.07       9.79     520.00     520.00   

       left_action  right_action  left_scored  right_scored    session  
count    100

In [39]:
# rows per session, we want them to be roughly similar in length
rows_per_session = data.groupby("session").size()
print("Rows per session:")
print(rows_per_session.to_string())
print(f"\nMin: {rows_per_session.min():,}  Max: {rows_per_session.max():,}  Mean: {rows_per_session.mean():,.0f}")

Rows per session:
session
0    12929
1     9598
2     5121
3     9506
4     8259
5    11799
6    16769
7     9118
8     6725
9    10674

Min: 5,121  Max: 16,769  Mean: 10,050


In [40]:
# action distribution for both paddles
# -1 = up, 0 = stay, 1 = down
action_labels = {-1: "up", 0: "stay", 1: "down"}

for side in ["left", "right"]:
    col = f"{side}_action"
    counts = data[col].value_counts().sort_index()
    total  = len(data)
    print(f"\n{side.upper()} paddle action distribution:")
    for action, count in counts.items():
        label = action_labels[action]
        pct   = count / total * 100
        bar   = "█" * int(pct / 2)
        print(f"  {label:>5} ({action:+d}): {count:>7,}  {pct:5.1f}%  {bar}")


LEFT paddle action distribution:
     up (-1):  19,468   19.4%  █████████
   stay (+0):  61,505   61.2%  ██████████████████████████████
   down (+1):  19,525   19.4%  █████████

RIGHT paddle action distribution:
     up (-1):  21,050   20.9%  ██████████
   stay (+0):  58,393   58.1%  █████████████████████████████
   down (+1):  21,055   21.0%  ██████████


In [41]:
# constants matching play.py
WIDTH     = 800
HEIGHT    = 600
PADDLE_H  = 80

def engineer_features(df):
    d = df.copy()

    # Paddle center y
    d["left_center_y"]  = d["left_y"]  + PADDLE_H / 2
    d["right_center_y"] = d["right_y"] + PADDLE_H / 2

    # Ball distance from each paddle center (signed: + means ball is below paddle)
    d["left_ball_dy"]   = d["ball_y"] - d["left_center_y"]
    d["right_ball_dy"]  = d["ball_y"] - d["right_center_y"]

    # Horizontal distance from ball to each paddle
    d["left_ball_dx"]   = d["ball_x"] - 30       # left paddle x is ~30
    d["right_ball_dx"]  = 758 - d["ball_x"]      # right paddle x is ~758

    # Normalize everything to [-1, 1] range
    d["ball_x_n"]         = d["ball_x"]         / WIDTH
    d["ball_y_n"]         = d["ball_y"]         / HEIGHT
    d["ball_vx_n"]        = d["ball_vx"]        / 14.0   # BALL_SPEED_MAX
    d["ball_vy_n"]        = d["ball_vy"]        / 14.0
    d["left_center_y_n"]  = d["left_center_y"]  / HEIGHT
    d["right_center_y_n"] = d["right_center_y"] / HEIGHT
    d["left_ball_dy_n"]   = d["left_ball_dy"]   / HEIGHT
    d["right_ball_dy_n"]  = d["right_ball_dy"]  / HEIGHT
    d["left_ball_dx_n"]   = d["left_ball_dx"]   / WIDTH
    d["right_ball_dx_n"]  = d["right_ball_dx"]  / WIDTH

    return d

data = engineer_features(data)

FEATURES = [
    "ball_x_n", "ball_y_n", "ball_vx_n", "ball_vy_n",
    "left_center_y_n", "right_center_y_n",
    "left_ball_dy_n", "left_ball_dx_n",
    "right_ball_dy_n", "right_ball_dx_n",
]

print(f"Feature count : {len(FEATURES)}")
print(data[FEATURES].describe().round(3))

Feature count : 10
         ball_x_n    ball_y_n   ball_vx_n   ball_vy_n  left_center_y_n  \
count  100498.000  100498.000  100498.000  100498.000       100498.000   
mean        0.503       0.481       0.004       0.002            0.498   
std         0.258       0.269       0.404       0.258            0.221   
min        -0.007       0.010      -0.783      -0.699            0.067   
25%         0.280       0.255      -0.381      -0.210            0.342   
50%         0.505       0.487       0.189       0.006            0.500   
75%         0.726       0.692       0.378       0.207            0.667   
max         1.007       0.990       0.719       0.699            0.933   

       right_center_y_n  left_ball_dy_n  left_ball_dx_n  right_ball_dy_n  \
count        100498.000      100498.000      100498.000       100498.000   
mean              0.506          -0.017           0.466           -0.025   
std               0.245           0.215           0.258            0.224   
min       

In [42]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

VAL_FRACTION = 0.2
BATCH_SIZE   = 256

all_sessions = sorted(data["session"].unique())
n_val        = max(1, int(len(all_sessions) * VAL_FRACTION))
val_sessions = all_sessions[-n_val:]
trn_sessions = all_sessions[:-n_val]

trn = data[data["session"].isin(trn_sessions)]
val = data[data["session"].isin(val_sessions)]

print(f"Train sessions : {trn_sessions}  ({len(trn):,} rows)")
print(f"Val   sessions : {val_sessions}  ({len(val):,} rows)")

Train sessions : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]  (83,099 rows)
Val   sessions : [np.int64(8), np.int64(9)]  (17,399 rows)


In [43]:
def make_xy(df, side):
    X = df[FEATURES].values.astype(np.float32)
    y = df[f"{side}_action"].values   # -1, 0, 1
    y = y + 1                         # remap to 0, 1, 2 for CrossEntropy
    return X, y.astype(np.int64)

X_trn_l, y_trn_l = make_xy(trn, "left")
X_trn_r, y_trn_r = make_xy(trn, "right")
X_trn = np.concatenate([X_trn_l, X_trn_r])
y_trn = np.concatenate([y_trn_l, y_trn_r])

X_val_l, y_val_l = make_xy(val, "left")
X_val_r, y_val_r = make_xy(val, "right")
X_val = np.concatenate([X_val_l, X_val_r])
y_val = np.concatenate([y_val_l, y_val_r])

classes = np.array([0, 1, 2])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_trn)
class_weights = torch.tensor(cw, dtype=torch.float32)

print(f"X_trn : {X_trn.shape}   y_trn : {y_trn.shape}")
print(f"X_val : {X_val.shape}   y_val : {y_val.shape}")
print("\nClass weights (0=up, 1=stay, 2=down):")
for label, w in zip(["up", "stay", "down"], cw):
    print(f"  {label:>4} : {w:.4f}")

X_trn : (166198, 10)   y_trn : (166198,)
X_val : (34798, 10)   y_val : (34798,)

Class weights (0=up, 1=stay, 2=down):
    up : 1.6799
  stay : 0.5538
  down : 1.6696


In [44]:
class PongDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

trn_loader = DataLoader(PongDataset(X_trn, y_trn), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(PongDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(trn_loader)}")
print(f"Val   batches : {len(val_loader)}")
print("\nDataLoaders ready")

Train batches : 650
Val   batches : 136

DataLoaders ready


In [47]:
import torch.nn as nn

class PongMLP(nn.Module):
    def __init__(self, input_dim=10, hidden=96, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 3),
        )

    def forward(self, x):
        return self.net(x)

model     = PongMLP()
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 10,659


In [48]:
EPOCHS = 40

train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for X_batch, y_batch in trn_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * len(y_batch)
        t_correct += (logits.argmax(1) == y_batch).sum().item()
        t_total   += len(y_batch)

    # --- val ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            v_loss    += loss.item() * len(y_batch)
            v_correct += (logits.argmax(1) == y_batch).sum().item()
            v_total   += len(y_batch)

    trn_loss = t_loss / t_total
    val_loss = v_loss / v_total
    trn_acc  = t_correct / t_total * 100
    val_acc  = v_correct / v_total * 100

    train_losses.append(trn_loss)
    val_losses.append(val_loss)
    train_accs.append(trn_acc)
    val_accs.append(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}  |  trn loss {trn_loss:.4f}  acc {trn_acc:.1f}%"
              f"  |  val loss {val_loss:.4f}  acc {val_acc:.1f}%")

print("\nDone.")

Epoch   1  |  trn loss 0.9706  acc 41.0%  |  val loss 0.9315  acc 44.9%
Epoch   5  |  trn loss 0.8998  acc 48.0%  |  val loss 0.8915  acc 50.4%
Epoch  10  |  trn loss 0.8818  acc 49.0%  |  val loss 0.8835  acc 49.1%
Epoch  15  |  trn loss 0.8732  acc 49.7%  |  val loss 0.8853  acc 49.5%
Epoch  20  |  trn loss 0.8684  acc 50.1%  |  val loss 0.8766  acc 49.2%
Epoch  25  |  trn loss 0.8638  acc 50.4%  |  val loss 0.8824  acc 49.1%
Epoch  30  |  trn loss 0.8620  acc 50.6%  |  val loss 0.8834  acc 49.2%
Epoch  35  |  trn loss 0.8589  acc 50.6%  |  val loss 0.8836  acc 49.0%
Epoch  40  |  trn loss 0.8559  acc 51.0%  |  val loss 0.8802  acc 49.7%

Done.


In [57]:
torch.save(model.state_dict(), "bc_model.pt")